# Search for given term
Input search term  
Checks disease, chemical, species files for synonyms  
Uses buffer to search through pubtator titles and save titles (and IDs) where titles contain words in synonyms

## Import packages 

In [2]:
import sqlite3
import gzip
import requests
import tarfile
import gzip 
import io
import json
from lxml import etree #both of these should be fine but aren't working 
from tqdm.auto import tqdm #"no module named"
import time
import sqlite3
from typing import Optional
import os
import sys
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

/home/zplumridge_smith_edu/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Synonym Config

In [3]:
URL = "https://ftp.ncbi.nlm.nih.gov/pub/lu/PubTator3/bioconcepts2pubtator3.gz"
SQLITE_PATH = "pubtator_synonyms.sqlite"
MEMBER_LIMIT = 2           # set to None to process all members
DOCS_COMMIT_BATCH = 200     # commit every N documents
PROCESS_ONLY_XML = False     # if True, skip non-XML members
SHOW_PROGRESS = True

In [4]:
CREATE_TABLES_SQL = """
CREATE TABLE IF NOT EXISTS pub2mesh (
    pmid    TEXT,
    mesh_id TEXT
);
CREATE TABLE IF NOT EXISTS mesh2bioconcept (
    mesh_id    TEXT,
    bioconcept TEXT
);
CREATE TABLE IF NOT EXISTS mesh2cellline (
    mesh_id    TEXT,
    cellline TEXT
);
CREATE TABLE IF NOT EXISTS mesh2chemical (
    mesh_id    TEXT,
    chemical TEXT
);
CREATE TABLE IF NOT EXISTS mesh2disease (
    mesh_id    TEXT,
    disease TEXT
);
CREATE TABLE IF NOT EXISTS mesh2gene (
    mesh_id    TEXT,
    gene TEXT
);
CREATE TABLE IF NOT EXISTS mesh2mutation (
    mesh_id    TEXT,
    mutation TEXT
);
CREATE TABLE IF NOT EXISTS mesh2relation (
    mesh_id    TEXT,
    relation TEXT
);
CREATE TABLE IF NOT EXISTS mesh2species (
    mesh_id    TEXT,
    species TEXT
);
-- bookkeeping: record processed member names so we can resume
CREATE TABLE IF NOT EXISTS processed_members (
    member_name TEXT PRIMARY KEY,
    processed_at REAL
);
"""


In [5]:
def init_db(path: str):
    conn = sqlite3.connect(path, timeout=60)
    cur = conn.cursor()
    cur.executescript(CREATE_TABLES_SQL)
    conn.commit()
    return conn

In [6]:
# Small XML helper utilities
def localname(tag: Optional[str]) -> Optional[str]:
    if tag is None:
        return None
    return tag.split("}")[-1] if "}" in tag else tag

def get_child_text(elem, child_name: str) -> Optional[str]:
    for ch in elem:
        if localname(ch.tag) == child_name:
            return (ch.text or "").strip()
    return None

def get_children(elem, child_name: str):
    for ch in elem:
        if localname(ch.tag) == child_name:
            yield ch

def infons_to_dict(parent_elem):
    d = {}
    for inf in get_children(parent_elem, "infon"):
        if "key" in inf.attrib:
            key = inf.attrib["key"]
            d[key] = (inf.text or "").strip()
        else:
            # fallback: accumulate unnamed infons
            v = (inf.text or "").strip()
            if v:
                d.setdefault("notes", []).append(v)
    return d

In [7]:
# Core: parse one archive member (BioC XML) incrementally and insert into DB
def parse_member_and_insert(member_fileobj, member_name: str, conn: sqlite3.Connection,
                            docs_commit_batch: int = DOCS_COMMIT_BATCH,
                            max_docs: Optional[int]=None,
                            show_progress: bool = SHOW_PROGRESS):
    """
    member_fileobj: binary file-like for the member bytes
    member_name: name string (used for bookkeeping)
    conn: sqlite connection
    max_docs: optional limit of documents to process for this member (useful for testing)
    Returns: number of documents processed
    """
    # synonyms files are gzipped but not tarred 
    # wrap gzip if member itself is gz inside the tar
    if member_name.endswith(".gz"):
        binstream = gzip.GzipFile(fileobj=member_fileobj)
    else:
        binstream = member_fileobj

    parser = etree.XMLParser(recover=True, huge_tree=True)
    context = etree.iterparse(binstream, events=("end",))

    cur = conn.cursor()
    docs = 0
    t0 = time.time()
    try:
        for event, elem in context: 
            print("reading event")
    
    finally:
        # final commit for this member
        conn.commit()
        try:
            context.close()
        except Exception:
            pass

    return docs

In [10]:
def stream_and_process(url: str, conn: sqlite3.Connection,
                           member_limit: Optional[int] = MEMBER_LIMIT,
                           docs_commit_batch: int = DOCS_COMMIT_BATCH,
                           process_only_xml: bool = PROCESS_ONLY_XML,
                           resume: bool = True):
    resp = requests.get(url, stream=True, timeout=60)
    #print(resp) >> <Response [200]>
    resp.raise_for_status()
    resp.raw.decode_content = True
    
    #file = open(fileobj=resp, "r")
    #line = file.readline()
    # while line:
    #     print(line.strip()) #strip removes newline chars
    #     line = file.readline()
    # file.close()
    with gzip.open(resp.raw, 'rb') as f:
        file_content = f.read()
        line = file_content
        while line:
            print(line.strip())
            line = file_content.readline()
        file_content.close()
    gzip.close()
        # makes kernel crash 
    # can I decrease the size of unzipping happening? use peek()?

In [13]:
# Stream the tar archive, process each member, and record progress for resume
def stream_tar_and_process(url: str, conn: sqlite3.Connection,
                           member_limit: Optional[int] = MEMBER_LIMIT,
                           docs_commit_batch: int = DOCS_COMMIT_BATCH,
                           process_only_xml: bool = PROCESS_ONLY_XML,
                           resume: bool = True):
    """
    Streams the tar.gz at url and processes members sequentially.
    If resume=True, members listed in processed_members table are skipped.
    """
    # Get set of already processed members
    cur = conn.cursor()
    if resume:
        cur.execute("SELECT member_name FROM processed_members")
        processed_set = set(r[0] for r in cur.fetchall())
    else:
        processed_set = set()

    resp = requests.get(url, stream=True, timeout=60)
    #print(resp) >> <Response [200]>
    resp.raise_for_status()
    resp.raw.decode_content = True
    # choose streaming mode; let tarfile detect compression
    tar = tarfile.open(fileobj=resp.raw, mode="r|*") #not tar files here
    # invalid header - bc not tar file!
    #file = open(fileobj=resp, "r")
    #line = file.readline()
    # while line:
    #     print(line.strip()) #strip removes newline chars
    #     line = file.readline()
    # file.close()

    members_done = 0
    total_docs = 0
    t0_all = time.time()

    try:
        for member in tar:
            if member_limit is not None and members_done >= member_limit:
                break
            if not member.isfile():
                continue
            name = member.name
            if resume and name in processed_set:
                if SHOW_PROGRESS:
                    print(f"Skipping already-processed member: {name}")
                members_done += 1
                continue
            # Optionally skip non-XML files
            if process_only_xml and not (name.lower().endswith(".xml") or name.lower().endswith(".xml.gz") or name.lower().endswith(".bioc") or name.lower().endswith(".bioc.gz")):
                if SHOW_PROGRESS:
                    print(f"Skipping non-XML member: {name}")
                members_done += 1
                # we don't mark non-XML members as processed to allow future runs to reconsider them
                continue

            if SHOW_PROGRESS:
                print(f"Processing member: {name}")

            fobj = tar.extractfile(member)
            if fobj is None:
                print(f"  [WARN] could not extract {name}")
                members_done += 1
                continue

            try:
                docs = parse_member_and_insert(fobj, name, conn, docs_commit_batch)
            except Exception as e:
                # On errors, commit what we have and raise or continue based on policy.
                conn.commit()
                print(f"  [ERROR] parsing member {name}: {e}", file=sys.stderr)
                # Option: mark as failed by not recording processed_members so you can retry later.
                # We'll re-raise to stop unless you prefer to continue
                raise
            finally:
                try:
                    fobj.close()
                except Exception:
                    pass

            # record that member completed
            cur.execute("INSERT OR REPLACE INTO processed_members(member_name, processed_at) VALUES (?, ?)", (name, time.time()))
            conn.commit()

            members_done += 1
            total_docs += docs
            if SHOW_PROGRESS:
                print(f"  finished member {name}: {docs} documents")

    finally:
        try:
            tar.close()
        except Exception:
            pass
        try:
            resp.close()
        except Exception:
            pass

    elapsed = time.time() - t0_all
    print(f"Completed {members_done} members, {total_docs} documents in {elapsed:.1f}s")

In [11]:
if __name__ == "__main__":
    searchTerm = input()
    print(searchTerm)
    conn = init_db(SQLITE_PATH)
    try:
        stream_and_process(URL, conn, member_limit=MEMBER_LIMIT)
    finally:
        conn.close()

 all


all


KeyboardInterrupt: 